# Arc-length pipeline: models -> prompts -> activations -> arc lengths

Runs every stage sequentially for each model in `MODELS`. Generated data is kept under `artifacts/<model name>/`; after each run, the model and tokenizer are released and that model repository is removed from the local Hugging Face download cache. Generated datasets, activation caches, surface artifacts, plots, and inference outputs are retained.

The cached layer is selected independently for every model as `floor(0.6 * num_hidden_layers)`.

In [ ]:
from pathlib import Path
import gc
import math
import sys
import traceback

import torch
from huggingface_hub import scan_cache_dir
from IPython.display import display
from transformers import AutoConfig

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'stakes_surface_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import (cache_activations, context_inference, prompt_datasets, severity_flipped_inference,
                     severity_inference, severity_pairwise_inference, severity_wording_inference)
from scripts.pipeline_config import NO_TIME_CORPORA, RunConfig, naming_convention_spec
from scripts.stakes_surface_pipeline import StakesSurfacePipeline

print('Repository:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

## Configuration

Add `(model_name, naming_convention)` pairs to `MODELS` in execution order. Use `"llama"` for models whose decoder blocks are named `model.layers.N`, and `"gemma4"` for Gemma 4 multimodal checkpoints. To support another convention, add its dotted config layer-count attribute and module-path template to `NAMING_CONVENTIONS` in `scripts/pipeline_config.py`; add a loader branch in `cache_activations.load_model` only if the architecture cannot use the standard causal-LM loader. Cache keys remain `layer_out/N` for downstream compatibility. For gated models, authenticate before starting. `BATCH_SIZE` applies to every model; lower it if needed. Compatible artifacts are reused unless `FORCE` is enabled.

In [ ]:
MODELS = [
    ('Qwen/Qwen3-4B-Instruct-2507', 'llama'),
    ('Qwen/Qwen3-8B', 'llama'),
    ('Qwen/Qwen3-14B', 'llama'),
    ('mistralai/Mistral-Small-3.1-24B-Instruct-2503', 'llama'),
    ('google/gemma-3-27b-it', 'gemma4'),
    ('Qwen/Qwen3-32B', 'llama'),
    ('google/gemma-4-31B-it', 'gemma4'),
]
BATCH_SIZE = 256
STAKES_MERGES = {
    'near_existential': 'existential',
    'medium_low': 'medium',
}
POSITION = -1                    # final prompt token
CORPORA = list(NO_TIME_CORPORA)  # quick smoke run: ['conversational_no_time']
FORCE = False                    # rebuild caches that fail fingerprint checks

## Pipeline helpers

Cache cleanup is repository-scoped: it removes all cached revisions of the completed model ID while allowing Hugging Face to preserve blobs shared by other repositories. It never targets this repository's `artifacts/` tree.

In [ ]:
def cached_layer_for_model(model_name, naming_convention):
    count_attribute, _ = naming_convention_spec(naming_convention)
    architecture = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    layer_config = architecture
    for attribute in count_attribute.split('.'):
        layer_config = getattr(layer_config, attribute, None)
        if layer_config is None:
            break
    num_layers = layer_config
    if not isinstance(num_layers, int) or num_layers < 1:
        raise ValueError(f'{model_name}: config has no positive integer {count_attribute}')
    layer = math.floor(0.6 * num_layers)
    if not 0 <= layer < num_layers:
        raise ValueError(f'{model_name}: computed invalid layer {layer} for {num_layers} layers')
    return num_layers, layer


def delete_local_model_data(model_name):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    cache = scan_cache_dir()
    revisions = [revision.commit_hash
                 for repo in cache.repos
                 if repo.repo_type == 'model' and repo.repo_id == model_name
                 for revision in repo.revisions]
    if not revisions:
        print(f'No local Hugging Face model cache remains for {model_name}.')
        return
    strategy = cache.delete_revisions(*revisions)
    print(f'Deleting {model_name} download cache ({strategy.expected_freed_size_str}) ...')
    strategy.execute()


def show_inference_result(label, result):
    csv_path, rows, diagnostics = result
    display(rows)
    display(diagnostics.groupby('coordinate_status').size().rename('rows').to_frame())
    display(diagnostics[['outside_saved_height_range', 'surface_extended', 'height_extrapolated']].sum().rename('rows').to_frame())
    display(diagnostics[['surface_projection_residual', 'slice_height_error']].describe())
    print(f'{label} CSV:', csv_path)


def process_model(model_name, naming_convention):
    num_layers, layer = cached_layer_for_model(model_name, naming_convention)
    config = RunConfig(model_name=model_name, naming_convention=naming_convention,
                       layer_component=f'layer_out/{layer}',
                       position=POSITION, batch_size=BATCH_SIZE, stakes_merges=STAKES_MERGES)
    print(f'\n=== {model_name}: caching layer floor(0.6 * {num_layers}) = {layer} ===')
    print(config.describe())
    print(f'Cached module: {config.layer_module_name} ({naming_convention})')

    model = tokenizer = None
    try:
        datasets = prompt_datasets.generate_datasets(config, CORPORA)
        prompt_datasets.preflight(datasets)
        model, tokenizer = cache_activations.load_model(config)
        print(f'Loaded {model_name} on {model.device}')
        cache_paths = cache_activations.run(config, datasets, model=model, tokenizer=tokenizer, force=FORCE)
        print(f'{len(cache_paths)} template caches in {config.activations_dir}')
    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pipeline = StakesSurfacePipeline(config)
    display(pipeline.load_caches())
    projected_rows = pipeline.fit_bcpc()
    bcpc = pipeline.bcpc
    print(f'{len(projected_rows):,} rows, {len(bcpc.projection["classes"])} merged classes')
    display(bcpc.class_table()); display(bcpc.variance_table())
    display(bcpc.centroids); display(bcpc.anchors)
    print(f'Weighted squared residual sum: {bcpc.weighted_residual_sum:.6g}')
    display(bcpc.spline_points)
    print(f's=0: spline endpoint associated with {bcpc.zero_anchor}')
    print(f'Total spline arc length: {bcpc.total_arc_length:.6g}')
    display(projected_rows[['stakes', 'spline_parameter', 'bcpc_arc_length', 'distance_to_spline']].head())
    display(pipeline.fit_pls()); display(pipeline.weight_summary)
    rotation_diagnostics, file_centroids = pipeline.rotate_plane()
    display(rotation_diagnostics); display(file_centroids)
    display(pipeline.fit_surface())
    display(pipeline.project_stated_rows())
    display(pipeline.build_slice_cache())
    mapping = pipeline.map_coordinates()
    for table in mapping.values():
        display(table)
    display(pipeline.all_rows[['source_file', 'stakes', 'bcpc_arc_length',
                               'arc_length_parallel', 'arc_length_orthogonal']].head())
    output_dir = pipeline.export(notebook='notebooks/arc_length_pipeline.ipynb')
    figures = pipeline.save_plots()
    for figure in figures.values():
        figure.show()

    show_inference_result('Severity', severity_inference.run(config, force=FORCE))
    show_inference_result('Flipped severity', severity_flipped_inference.run(config, force=FORCE))
    show_inference_result('Pairwise severity', severity_pairwise_inference.run(config, force=FORCE))
    show_inference_result('Severity wording', severity_wording_inference.run(config, force=FORCE))
    show_inference_result('Context variation', context_inference.run(config, force=FORCE))
    return output_dir

## Run every model sequentially

A failed model does not prevent later entries from running. Its traceback is recorded, local model data is still cleaned up, and the cell raises after printing the complete status table.

In [ ]:
if not MODELS:
    raise ValueError('MODELS must contain at least one (model_name, naming_convention) pair.')
for entry in MODELS:
    if not isinstance(entry, (tuple, list)) or len(entry) != 2 or not all(isinstance(value, str) and value for value in entry):
        raise ValueError('Each MODELS entry must be a (model_name, naming_convention) pair of nonempty strings.')
    naming_convention_spec(entry[1])
model_names = [name for name, _ in MODELS]
if len(model_names) != len(set(model_names)):
    raise ValueError('MODELS contains duplicate model IDs.')

run_results = {}
for model_name, naming_convention in MODELS:
    try:
        output_dir = process_model(model_name, naming_convention)
        run_results[model_name] = {'status': 'complete', 'output_dir': str(output_dir)}
    except Exception as exc:
        run_results[model_name] = {'status': 'failed',
                                   'error': f'{type(exc).__name__}: {exc}',
                                   'traceback': traceback.format_exc()}
        print(run_results[model_name]['traceback'])
    finally:
        try:
            delete_local_model_data(model_name)
        except Exception as cleanup_error:
            cleanup_message = f'{type(cleanup_error).__name__}: {cleanup_error}'
            run_results.setdefault(model_name, {'status': 'failed'})['cleanup_error'] = cleanup_message
            print(f'WARNING: cleanup failed for {model_name}: {cleanup_message}')

display(run_results)
failures = {name: result for name, result in run_results.items() if result['status'] != 'complete'}
cleanup_failures = {name: result for name, result in run_results.items() if 'cleanup_error' in result}
if failures or cleanup_failures:
    raise RuntimeError(f'Pipeline failures: {list(failures)}; cleanup failures: {list(cleanup_failures)}')

## Reusing an exported surface

```python
from scripts.stakes_surface_bundle import load_surface_bundle, project_saved_bcpc
arrays, metadata, saved_coordinates = load_surface_bundle(config.surface_dir)
bcpc_projection = project_saved_bcpc(activation_batch, arrays, metadata)
pls_scores = (activation_batch - arrays['pls_mean']) @ arrays['pls_rotations']
new_surface_coordinates = saved_coordinates.map_points(pls_scores[:, :3])
```